In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np


importing tools to do the train test split, make the pipeline, column transformer to easily convert the categorical and numerical features into onehot encoding and scaling repectively

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

lets load the csv file now

In [ ]:
df=pd.read_csv("bank-additional-full.csv", sep=';')

In [ ]:
df.shape

In [ ]:
df.info()

so there are no missing values at all

In [ ]:
df.duplicated().sum()

there are 12 duplicated rows in here. lets see them

In [ ]:
duplicates = df[df.duplicated]

In [ ]:
duplicates

nothing wrong with the duplicated rows so ill drop them. 

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.shape

now lets see if our data is imbalanced or not 

In [ ]:
df["y"].value_counts()

There are 36537 nos and only 4639 yes'. data is imbalanced

i will handle the imbalanced dataset after splitting; not now. 

In [ ]:
df.describe()

now i will look at the features that we have : 
"age" "job" "marital" "education" "default" "housing" "loan" "contact" "month" "day_of_week" "duration" "campaign" "pdays" "previous" "poutcome" "emp.var.rate" "cons.price.idx" "cons.conf.idx" "euribor3m" "nr.employed" "y"

here duration represents the length of call that happened when the customer was called last time. here we are deciding  which customers to call before making the call. we cannot know the duration of the call before actually making it. 

also it is highly misleading, because the duration with 0 seconds willl always have answer 'no'

therefore we drop duration

In [ ]:
df = df.drop(columns=['duration'])

In [ ]:
for col in categorical_cols:
    print(df[col].value_counts())
    print('\n')

there are unknown values here. but i will consider them as a separate cztegory rather than removing them or imputing anything else. 

pdays:

In [ ]:
print(df['pdays'].value_counts())

999 pdays means customers were not contacted before. and there are ALOT of them

so what i will do is i will add onemore column named contactedaBefore which will be a binary column. it will have 1 if contacted before else 0 (pdays=999)

In [ ]:
df['contacted_before'] = np.where(df['pdays'] == 999, 0, 1)


after doing this , we have separated the cusstomers into contactedbefore or not. but we need to chnage our og pdays column as well. we will change all the 999 values to 0. and keeping rest of it as it is. so the workflow will look something like: model will look if it was contacted before, from the new added column. if yes, thenn it will look howmany days ago from pdays column

In [ ]:

df['pdays'] = np.where(df['pdays'] == 999, 0, df['pdays'])
df['y'] = df['y'].map({'yes': 1, 'no': 0})



In [ ]:
df.shape


In [ ]:
df.iloc[0]

In [ ]:
df.iloc[-1]


now the given dataset is order chronologically meaning the customers who were called first are literaly placed above. so we will not split the data in random order but instead split as first 80% and last 20%. basically we will train on more ""historical data"". 

also our y is in string format eith yes and no. so we convert it into 1 and 0

In [ ]:

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]


X_train = train_df.drop(columns=['y'])
y_train = train_df['y']

X_test = test_df.drop(columns=['y'])
y_test = test_df['y']

#updating our numeric cols and categorical cols 
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")




using  columntransformer to do one hot encoding to our categorical features. and scale our numeric cols

In [ ]:
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])



so now i will make our first pipeline. this will be our baseline. we still havent fixed our imbalanced dataset. so keeping it as our baseline will help us on next steps. 

we splitted dataset into 80-20 ratio. 

we added a contacted_before column to our dataframe. we converted string y into binary 1 and 0. 

we made a preprocessor to convert the categorical into one hot encode and numeric into proper scaling. 

In [ ]:
from sklearn.linear_model import LogisticRegression


baseline_model = Pipeline([
    ('preprocessor', preprocessor),
  
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)) 
])


baseline_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score


y_pred = baseline_model.predict(X_test)


y_prob = baseline_model.predict_proba(X_test)[:, 1] 


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report (Precision, Recall, F1):")
print(classification_report(y_test, y_pred))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC Score (Average Precision): {average_precision_score(y_test, y_prob):.4f}")

yes its terrible. since more nos are there in our dataset, it highly biased there. giving high recall and high f1 for class 0. 

now we will add class weight=balanced in our logistic regression model and see where that takes us. 

In [ ]:

balanced_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)) 
])

balanced_model.fit(X_train, y_train)


y_pred_bal = balanced_model.predict(X_test)
y_prob_bal = balanced_model.predict_proba(X_test)[:, 1]



print("--- Balanced Model Evaluation ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_bal))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bal))
print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_prob_bal):.4f}")
print(f"PR-AUC Score (Average Precision): {average_precision_score(y_test, y_prob_bal):.4f}")

so it is catching the yes now because of our penalty but its givin too much yes making our precision down. 

lets see precision@500 and lift@500

In [153]:



base_rate = y_test.mean() 
expected_random_subscribers = base_rate * 500

print("1. Baseline (Blind Luck)")
print(f" this is base rate{base_rate:.4f}")
print(f"If a call center dialed 500 random people, they'd find ~{expected_random_subscribers:.1f} subscribers.\n")



ranking_df = pd.DataFrame({
    'Actual_Answer': y_test,
    'Probability_Score': y_prob_bal # Probabilities from the balanced model
})

ranking_df = ranking_df.sort_values(by='Probability_Score', ascending=False)


top_500_leads = ranking_df.head(500)


actual_subscribers_found = top_500_leads['Actual_Answer'].sum()
precision_at_500 = actual_subscribers_found / 500


lift_at_500 = precision_at_500 / base_rate

print(" 2. Our Balanced Model's Performance ")
print(f"Subscribers found in Top 500 calls: {actual_subscribers_found}")
print(f"Precision@500: {precision_at_500:.4f}")
print(f"Lift@500: {lift_at_500:.2f}x")
print(f"(This means our model is {lift_at_500:.2f} times more effective than calling at random!)")

1. Baseline (Blind Luck)
 this is base rate0.3083
If a call center dialed 500 random people, they'd find ~154.1 subscribers.

 2. Our Balanced Model's Performance 
Subscribers found in Top 500 calls: 301
Precision@500: 0.6020
Lift@500: 1.95x
(This means our model is 1.95 times more effective than calling at random!)


so we can conclude that our baseline model has got around 301 subscribers out of 500.

so now we know what this simple logistic regression with balanced data can give us. now we will look into other algorithms to see if we can improve from here. 60% is the % to beat.


In [154]:
from sklearn.ensemble import RandomForestClassifier

 # we start off with random forest. 

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])


rf_model.fit(X_train, y_train)


y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]


print("Random Forest Evaluation ")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_prob_rf):.4f}")


rf_ranking = pd.DataFrame({'Actual': y_test, 'Prob': y_prob_rf}).sort_values(by='Prob', ascending=False)
rf_top500 = rf_ranking.head(500)
rf_subscribers = rf_top500['Actual'].sum()
rf_prec_500 = rf_subscribers / 500
rf_lift_500 = rf_prec_500 / base_rate

print(f"\nSubscribers in Top 500: {rf_subscribers} / 500")
print(f"Precision@500: {rf_prec_500:.4f}")
print(f"Lift@500: {rf_lift_500:.2f}x")

Random Forest Evaluation 
Confusion Matrix:
[[3847 1850]
 [1078 1461]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.68      0.72      5697
           1       0.44      0.58      0.50      2539

    accuracy                           0.64      8236
   macro avg       0.61      0.63      0.61      8236
weighted avg       0.68      0.64      0.66      8236

ROC-AUC: 0.6815
PR-AUC: 0.4676

Subscribers in Top 500: 309 / 500
Precision@500: 0.6180
Lift@500: 2.00x


alright alright alright 

we did better here. but not by much. we got 61.80% and its doing 2x better than randomly selecting 500 calls. so yeah. 

In [155]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(
        class_weight='balanced',
        max_iter=200,
        random_state=42
    ))
])


gb_model.fit(X_train, y_train)


y_pred_gb = gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]


print(" Gradient Boosting Evaluation ")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))
print(f"PR-AUC: {average_precision_score(y_test, y_prob_gb):.4f}")


gb_ranking = pd.DataFrame({'Actual': y_test, 'Prob': y_prob_gb}).sort_values(by='Prob', ascending=False)
gb_top500 = gb_ranking.head(500)
gb_subscribers = gb_top500['Actual'].sum()

print(f"\nSubscribers in Top 500: {gb_subscribers} / 500")
print(f"Precision@500: {gb_subscribers / 500:.4f}")
print(f"Lift@500: {(gb_subscribers / 500) / base_rate:.2f}x")

 Gradient Boosting Evaluation 
Confusion Matrix:
[[4012 1685]
 [1399 1140]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.70      0.72      5697
           1       0.40      0.45      0.43      2539

    accuracy                           0.63      8236
   macro avg       0.57      0.58      0.57      8236
weighted avg       0.64      0.63      0.63      8236

PR-AUC: 0.4043

Subscribers in Top 500: 234 / 500
Precision@500: 0.4680
Lift@500: 1.52x


this performed poorly. boosting is out of question 